> 在线讲义：[第 1 章 · 总图：一台机器人 = 一组快慢嵌套的闭环](https://wikieden.github.io/robot-architecture/lessons/01-layered-loops.html) · 本书《机器人软硬件架构》

# 第 1 章 · 总图：一台机器人 = 一组快慢嵌套的闭环

配套讲义：仓库里 `robot-system-architecture/notes/lessons/01-layered-loops.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1、3 步 每层多快 | $T = 1/f$ | `period_ms(f)` | 20 kHz → 0.05 ms；10 Hz → 100 ms |
| 第 2 步 差 3 个数量级 | 100 ms ÷ 100 µs | `orders_of_magnitude` | 1000 倍 = 3 个数量级 |
| 第 4 步 中间 99 次 | 100 ms ÷ 1 ms | `steps_between(0.1, 0.001)` | 100 次，99 次靠插补 |
| 第 4 步 一阶追赶 | $x_{k+1} = x_k + \alpha(x^* - x_k)$ | `track(1.0, alpha, steps)` | α = 0.5 第 10 步 0.999；α = 1.5 来回过冲 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
LAYERS = {
    "L4 任务规划": 1.0,
    "L3 感知": 30.0,
    "L2 策略": 10.0,
    "L1 关节控制": 500.0,
    "L0 电机驱动": 20_000.0,
}

**函数 `period_ms`**：周期 T = 1 / f，换成毫秒。

In [ ]:
def period_ms(f_hz: float) -> float:
    """周期 T = 1 / f，换成毫秒。"""
    return 1000.0 / f_hz

**函数 `orders_of_magnitude`**：两个时间差了几个数量级：log10(慢 / 快)。

In [ ]:
def orders_of_magnitude(slow_s: float, fast_s: float) -> float:
    """两个时间差了几个数量级：log10(慢 / 快)。"""
    return float(np.log10(slow_s / fast_s))

**函数 `steps_between`**：慢层每给一次目标，快层要执行几次（含给新目标那一次）。

In [ ]:
def steps_between(t_slow_s: float, t_fast_s: float) -> int:
    """慢层每给一次目标，快层要执行几次（含给新目标那一次）。"""
    return round(t_slow_s / t_fast_s)

**函数 `track`**：一阶追赶 x_{k+1} = x_k + α (x* − x_k)，返回 x_0 … x_steps。

In [ ]:
def track(target: float, alpha: float, steps: int, x0: float = 0.0) -> np.ndarray:
    """一阶追赶 x_{k+1} = x_k + α (x* − x_k)，返回 x_0 … x_steps。"""
    xs = [x0]
    for _ in range(steps):
        xs.append(xs[-1] + alpha * (target - xs[-1]))
    return np.array(xs)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 / 第三步：每层的周期 T = 1/f")
for name, f in LAYERS.items():
    print(f"  {name:<8} {f:>8.0f} Hz → 周期 {period_ms(f):>8.3f} ms")

In [ ]:
print("\n第二步：为什么不能一层到底——电流环和 VLA 差几个数量级")
t_current = 100e-6  # 电流环 10 kHz → 100 µs（讲义 25–100 µs 取慢的一头）
t_vla = 100e-3  # VLA 一次推理按 100 ms 算
print(f"  VLA 100 ms ÷ 电流环 100 µs = {t_vla / t_current:.0f} 倍 = {orders_of_magnitude(t_vla, t_current):.0f} 个数量级")

In [ ]:
print("\n第四步：插补——L2 每 100 ms 给一个目标，L1 每 1 ms 要一个指令")
n = steps_between(0.1, 0.001)
print(f"  L1 每个目标要跑 {n} 次，其中 {n - 1} 次要靠插补补点")

In [ ]:
print("\n第四步：一阶追赶 x_{k+1} = x_k + α (x* − x_k)，目标 x* = 1")
for alpha in (0.1, 0.5):
    xs = track(1.0, alpha, 10)
    print(f"  α = {alpha}: 前 5 步 {np.round(xs[:6], 3).tolist()}  第 10 步 {xs[10]:.3f}")
xs = track(1.0, 1.5, 6)
print(f"  α = 1.5（太大）: {np.round(xs, 3).tolist()} ← 过冲、来回抖")
print("  误差每步乘 (1 − α)：α = 0.5 时第 10 步误差 = 0.5^10 =", round(0.5**10, 5))

## 3. 代码里才看得到的

- 误差每一步乘 $(1-\alpha)$，所以第 k 步误差是 $(1-\alpha)^k$：α = 0.5 时第 10 步误差 $0.5^{10} \approx 0.001$。测试就是这么核对的。
- α 超过 1 时 $(1-\alpha)$ 是负数，误差每步变号：1.5、0.75、1.125……这就是讲义说的"太大会过冲、抖动"。

## 4. 自己改一改

- 把 `LAYERS` 里的 L2 改成 2 Hz，看周期变成多少，对照动画里把"策略频率"拖到 2 Hz 的样子。
- 把 α 改成 0.9、1.9、2.1，看哪个还能收敛。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 大脑给目标，脊髓追目标；越往下越快、越简单、越不能迟到。
> 每一层软件都坐在一块硬件上，层和层之间是一条总线加一种消息。